# Cat and Dog Image Classifier  🐱🐶

**freeCodeCamp - Machine Learning with Python project**

A convolutional neural network (TensorFlow 2 / Keras) that tells cats and dogs
apart. It is trained on 2,000 images, validated on 1,000 images and finally
scores itself on the 50 unlabelled **test** images.

Target of the challenge: **at least 63%** of the 50 test images classified
correctly (extra credit for **70%+**).

### How to use this notebook
1. `Runtime -> Run all` (a free GPU is not required, but it makes training fast).
2. Cell 2 downloads the data set automatically (~230 MB, takes a minute).
3. The last cell prints the percentage of correct answers and whether you passed.
4. `File -> Save a copy in Drive`, then share the Colab link with
   *"Anyone with the link"* and submit that link on freeCodeCamp.

### What the pipeline does
| Step | Idea |
| ---- | ---- |
| 3 | read images from disk in batches, rescale pixels from 0-255 to 0-1 |
| 5 | data augmentation: rotate / shift / shear / zoom / flip every training image |
| 7 | 4 x (Conv2D + MaxPooling2D) + Flatten + Dense(512, ReLU) + Dropout + Dense(1, sigmoid) |
| 8 | train with Adam (lr = 1e-4) and binary cross-entropy |
| 10 | predict a dog-probability for each of the 50 test images |
| 11 | compare with the answer key, print accuracy / pass-fail |


In [ ]:
try:
  # This command only in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Get project files
!wget -q https://cdn.freecodecamp.org/project-data/cats-and-dogs/cats_and_dogs.zip

!unzip -q -o cats_and_dogs.zip

PATH = 'cats_and_dogs'

train_dir = os.path.join(PATH, 'train')
validation_dir = os.path.join(PATH, 'validation')
test_dir = os.path.join(PATH, 'test')

# Get number of files in each directory. The train and validation directories
# each have the subdirecories "dogs" and "cats".
total_train = sum([len(files) for r, d, files in os.walk(train_dir)])
total_val = sum([len(files) for r, d, files in os.walk(validation_dir)])
total_test = len(os.listdir(test_dir))

# Variables for pre-processing and training.
batch_size = 64      # smaller batches -> more gradient updates per epoch
epochs = 30          # the default was 15; a few more epochs help accuracy
IMG_HEIGHT = 150
IMG_WIDTH = 150

---
## Cell 3 - the three image generators

`ImageDataGenerator(rescale=1./255)` converts the 0-255 pixel values into
floating point tensors between 0 and 1.

`flow_from_directory` reads the pictures in batches of `batch_size` and
`class_mode='binary'` produces one 0/1 label per image (the folders are read in
alphabetical order, so **cat = 0, dog = 1**).

The `test` folder has *no* sub-folders, so we pass `classes=['.']`, which makes
`flow_from_directory` treat the folder itself as the single class. `shuffle=False`
keeps the 50 test images in the exact order the grader expects.

Expected output:

```
Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
Found 50 images belonging to 1 class.
```

In [ ]:
# 3
train_image_generator = ImageDataGenerator(rescale=1./255)
validation_image_generator = ImageDataGenerator(rescale=1./255)
test_image_generator = ImageDataGenerator(rescale=1./255)

train_data_gen = train_image_generator.flow_from_directory(
    batch_size=batch_size,
    directory=train_dir,
    shuffle=True,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary')

val_data_gen = validation_image_generator.flow_from_directory(
    batch_size=batch_size,
    directory=validation_dir,
    shuffle=False,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary')

# No sub-folders inside test/ -> classes=['.']
test_data_gen = test_image_generator.flow_from_directory(
    batch_size=batch_size,
    directory=test_dir,
    classes=['.'],
    shuffle=False,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None)

In [ ]:
# 4
def plotImages(images_arr, probabilities = False):
    fig, axes = plt.subplots(len(images_arr), 1, figsize=(5,len(images_arr) * 3))
    if probabilities is False:
      for img, ax in zip( images_arr, axes):
          ax.imshow(img)
          ax.axis('off')
    else:
      for img, probability, ax in zip( images_arr, probabilities, axes):
          ax.imshow(img)
          ax.axis('off')
          if probability > 0.5:
              ax.set_title("%.2f" % (probability*100) + "% dog")
          else:
              ax.set_title("%.2f" % ((1-probability)*100) + "% cat")
    plt.show()

sample_training_images, _ = next(train_data_gen)
plotImages(sample_training_images[:5])


---
## Cell 5 - data augmentation

With only 2,000 training images the network memorises them quickly
(over-fitting). Instead of collecting more photos we synthesise new ones on the
fly with **six** random transformations: rotation, horizontal & vertical shift,
shear, zoom and horizontal flip. `rescale` stays the same as before, and
`fill_mode='nearest'` fills the empty corners created by the shifts.

In [ ]:
# 5
train_image_generator = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest')

In [ ]:
# 6
train_data_gen = train_image_generator.flow_from_directory(batch_size=batch_size,
                                                     directory=train_dir,
                                                     target_size=(IMG_HEIGHT, IMG_WIDTH),
                                                     class_mode='binary')

augmented_images = [train_data_gen[0][0][0] for i in range(5)]

plotImages(augmented_images)

---
## Cell 7 - the convolutional neural network

Four convolutional blocks (`Conv2D` + `MaxPooling2D`) learn increasingly
abstract features (edges -> fur/ears -> faces), then a fully connected layer
activated by **ReLU** makes the decision. `Dropout(0.5)` randomly switches off
half of those neurons while training, which is the second weapon against
over-fitting. The final layer is a single neuron with a **sigmoid** activation,
i.e. the probability that the image is a dog (`> 0.5` -> dog).

In [ ]:
# 7
model = Sequential()

model.add(Conv2D(32, (3, 3), activation='relu',
                 input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)))
model.add(MaxPooling2D(2, 2))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))

model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

---
## Cell 8 - training

31 steps per epoch x 30 epochs, Adam with a small learning rate (1e-4) so the
network does not jump around, and the validation set is evaluated after every
epoch so we can watch for over-fitting.

In [ ]:
# 8
history = model.fit(
    train_data_gen,
    steps_per_epoch=total_train // batch_size,
    epochs=epochs,
    validation_data=val_data_gen,
    validation_steps=total_val // batch_size)

In [ ]:
# 9
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

---
## Cell 10 - predict the 50 unseen test images

First collect the test images in the order the generator produces them
(`shuffle=False`, so this is the order of the answer key), then ask the model
for the probability that each one is a dog. Images above 50% are labelled
`% dog`, the others `% cat`.

In [ ]:
# 10 - collect the 50 test images (unlabelled) in generator order
test_images = np.concatenate([batch for batch in test_data_gen], axis=0)
print('test images:', test_images.shape)

In [ ]:
# 10 - probability that each test image is a dog (0 = cat, 1 = dog)
probabilities = model.predict(test_data_gen)
probabilities = [float(p[0]) for p in probabilities]

plotImages(test_images, probabilities)

---
## Cell 11 - did we pass?

Compares the rounded predictions with the answer key.

In [ ]:
# 11
answers =  [1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0,
            1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0,
            1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1,
            1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 
            0, 0, 0, 0, 0, 0]

correct = 0

for probability, answer in zip(probabilities, answers):
  if round(probability) == answer:
    correct +=1

percentage_identified = (correct / len(answers)) * 100

passed_challenge = percentage_identified >= 63

print(f"Your model correctly identified {round(percentage_identified, 2)}% of the images of cats and dogs.")

if passed_challenge:
  print("You passed the challenge!")
else:
  print("You haven't passed yet. Your model should identify at least 63% of the images. Keep trying. You will get it!")